# 03 CleanData

Notebook นี้ใช้สำหรับ clean ข้อมูลต่อจากไฟล์ transformed

In [107]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [108]:
import pandas as pd

from src.utils.paths import INTERIM_DATA_DIR

INTERIM_DATA_DIR

WindowsPath('D:/ML/data/interim')

## Load Transformed Data

เริ่มจากไฟล์ `vw_timestamp_dashboard_transformed.csv` ที่ได้จากขั้น transform

In [109]:
source_path = INTERIM_DATA_DIR / 'vw_timestamp_dashboard_transformed.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f'source: {source_path}')
print(f'shape before clean: {df.shape}')
df.head()

source: D:\ML\data\interim\vw_timestamp_dashboard_transformed.csv
shape before clean: (10000, 40)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,ACCESSORIESSapAmount,TruckOverTimeName,TruckOverTimeRemark,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd,Unnamed: 39
0,SB1,Walk in,2026-04-10 10:01:58,9,เทรเลอร์,71-2054,SB1PL260410025,บ.ส.เอื้อสุข จก.,2026-04-10 10:02:03,N,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SB1,Walk in,2026-04-10 09:58:49,8,4 ล้อ,700-8513,SB1PL260410024,มบูเครือวัลย์ Lake and Park ลำผักชี,2026-04-10 09:59:06,N,...,34,NaN,NaN,NaN,NaN,2026-04-10 10:13:25,2026-04-10 10:20:44,2026-04-10 10:11:37,NaN,NaN
2,SB1,Walk in,2026-04-10 09:57:56,7,6 ล้อ,89-4045,SB1PL260410023,นครปฐม บางเลน,2026-04-10 09:57:58,N,...,0,NaN,NaN,NaN,NaN,2026-04-10 09:58:32,2026-04-10 09:59:48,NaN,NaN,NaN
3,SB1,Walk in,2026-04-10 09:56:17,6,เทรเลอร์,80-8223,SB1PL260410022,บ.อุดมชัยกู๊ดโฮมสมุทรสงคราม จก.,2026-04-10 09:56:23,N,...,0,NaN,NaN,2026-04-10 10:19:38,2026-04-10 10:20:33,2026-04-10 10:00:25,2026-04-10 10:08:47,NaN,NaN,NaN
4,SB1,Walk in,2026-04-10 09:48:27,5,6 ล้อ,89-4045,SB1PL260410021,บ.จำหน่ายวัตถุก่อสร้าง จก.,2026-04-10 09:48:32,N,...,70,NaN,NaN,NaN,NaN,2026-04-10 09:49:36,NaN,2026-04-10 10:05:27,2026-04-10 10:10:23,NaN


## Check Before Clean

เช็กคอลัมน์ `Unnamed: 39` และกระจายค่าของ `PackListStatus` ก่อนเริ่มลบ

In [110]:
print('has Unnamed: 39:', 'Unnamed: 39' in df.columns)
df['PackListStatus'].value_counts(dropna=False)

has Unnamed: 39: True


PackListStatus
OPERATORCOMPLETED    9654
REMOVE                303
WAITCHECKER            32
Loading                 6
CHECKERASSIGN           5
Name: count, dtype: int64

## Step 1: ลบคอลัมน์ `Unnamed: 39` , `TruckOverTimeName` , `TruckOverTimeRemark`

ลบคอลัมน์ที่ไม่ต้องใช้ก่อน

In [111]:
df_clean = df.copy()

columns_to_drop = ['Unnamed: 39', 'TruckOverTimeName', 'TruckOverTimeRemark']
existing_columns_to_drop = [col for col in columns_to_drop if col in df_clean.columns]
df_clean = df_clean.drop(columns=existing_columns_to_drop)

print(f'dropped columns: {existing_columns_to_drop}')
print(f'shape after dropping: {df_clean.shape}')
df_clean.columns.tolist()[-10:]

dropped columns: ['Unnamed: 39', 'TruckOverTimeName', 'TruckOverTimeRemark']
shape after dropping: (10000, 37)


['PRESTIGEFittingSapAmount',
 'NEUSTILEFittingSapAmount',
 'DURAFittingSapAmount',
 'ACCESSORIESSapAmount',
 'TileStart',
 'TileEnd',
 'FittingStart',
 'FittingEnd',
 'AccStart',
 'AccEnd']

## Step 2: เก็บค่า `OPERATORCOMPLETED`

เอาเฉพาะแถวที่ `PackListStatus` เท่ากับ `OPERATORCOMPLETED`

In [112]:
rows_before = len(df_clean)
df_clean = df_clean.loc[df_clean['PackListStatus'] == 'OPERATORCOMPLETED'].copy()
rows_after = len(df_clean)

print(f'rows before filter: {rows_before:,}')
print(f'rows after filter: {rows_after:,}')
print(f'rows removed: {rows_before - rows_after:,}')

rows before filter: 10,000
rows after filter: 9,654
rows removed: 346


In [113]:
df_clean['PackListStatus'].value_counts(dropna=False)

PackListStatus
OPERATORCOMPLETED    9654
Name: count, dtype: int64

## Step 3: เก็บ Timestamp Columns

เก็บเฉพาะแถวที่ timestamp ทั้ง 5 ช่องไม่ว่าง อยู่วันเดียวกัน และค่าเวลาไม่ซ้ำกันในแถวเดียวกัน

In [114]:
required_timestamp_columns = [
    'OperatorCarConfirm',
    'CarConfirm',
    'FirstPostPallet',
    'LastPostPallet',
    'PostingTime',
]

timestamp_check = df_clean[required_timestamp_columns].copy()
for col in required_timestamp_columns:
    timestamp_check[col] = pd.to_datetime(timestamp_check[col], errors='coerce')

timestamp_check.isna().sum()

OperatorCarConfirm    480
CarConfirm              0
FirstPostPallet       146
LastPostPallet        146
PostingTime            10
dtype: int64

In [115]:
rows_before = len(df_clean)

for col in required_timestamp_columns:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

non_null_mask = df_clean[required_timestamp_columns].notna().all(axis=1)
same_day_mask = df_clean.loc[non_null_mask, required_timestamp_columns].apply(
    lambda col: col.dt.date
).nunique(axis=1).eq(1)

same_day_index = df_clean.loc[non_null_mask].index[same_day_mask]
same_day_df = df_clean.loc[same_day_index].copy()
distinct_timestamp_mask = same_day_df[required_timestamp_columns].nunique(axis=1).eq(len(required_timestamp_columns))

df_clean = same_day_df.loc[distinct_timestamp_mask].copy()
rows_after = len(df_clean)

rows_after_non_null_same_day = len(same_day_df)
rows_removed_duplicate_timestamps = rows_after_non_null_same_day - rows_after

print(f'rows before timestamp filter: {rows_before:,}')
print(f'rows after non-null and same-day filter: {rows_after_non_null_same_day:,}')
print(f'rows removed because duplicate timestamps: {rows_removed_duplicate_timestamps:,}')
print(f'rows after timestamp filter: {rows_after:,}')
print(f'rows removed total: {rows_before - rows_after:,}')

rows before timestamp filter: 9,654
rows after non-null and same-day filter: 8,998
rows removed because duplicate timestamps: 1,381
rows after timestamp filter: 7,617
rows removed total: 2,037


## Step 4: Check Timestamp Order

ตรวจลำดับเวลาว่าต้องเรียงจาก `OperatorCarConfirm <= CarConfirm <= FirstPostPallet <= LastPostPallet <= PostingTime`

In [116]:
time_order_summary = pd.DataFrame(
    {
        'rule': [
            'OperatorCarConfirm <= CarConfirm',
            'CarConfirm <= FirstPostPallet',
            'FirstPostPallet <= LastPostPallet',
            'LastPostPallet <= PostingTime',
        ],
        'violated_rows': [
            (df_clean['OperatorCarConfirm'] > df_clean['CarConfirm']).sum(),
            (df_clean['CarConfirm'] > df_clean['FirstPostPallet']).sum(),
            (df_clean['FirstPostPallet'] > df_clean['LastPostPallet']).sum(),
            (df_clean['LastPostPallet'] > df_clean['PostingTime']).sum(),
        ],
    }
)

time_order_summary

,rule,violated_rows
0,OperatorCarConfirm <= CarConfirm,98
1,CarConfirm <= FirstPostPallet,0
2,FirstPostPallet <= LastPostPallet,0
3,LastPostPallet <= PostingTime,0


## Step 5: Remove Negative Durations

คำนวณ duration ชั่วคราวเพื่อตรวจแถวที่เวลาติดลบ แล้วลบแถวนั้นออก

In [117]:
duration_check = pd.DataFrame(index=df_clean.index)
duration_check['wait_call_min'] = (df_clean['CarConfirm'] - df_clean['OperatorCarConfirm']).dt.total_seconds() / 60
duration_check['prepare_loading_min'] = (df_clean['FirstPostPallet'] - df_clean['CarConfirm']).dt.total_seconds() / 60
duration_check['loading_time_min'] = (df_clean['LastPostPallet'] - df_clean['FirstPostPallet']).dt.total_seconds() / 60
duration_check['close_job_min'] = (df_clean['PostingTime'] - df_clean['LastPostPallet']).dt.total_seconds() / 60

negative_duration_summary = pd.DataFrame(
    {
        'duration_name': duration_check.columns,
        'negative_rows': [(duration_check[col] < 0).sum() for col in duration_check.columns],
    }
)

negative_duration_summary

,duration_name,negative_rows
0,wait_call_min,98
1,prepare_loading_min,0
2,loading_time_min,0
3,close_job_min,0


In [118]:
rows_before = len(df_clean)
negative_duration_mask = (duration_check < 0).any(axis=1)
df_clean = df_clean.loc[~negative_duration_mask].copy()
rows_after = len(df_clean)

print(f'rows before negative duration filter: {rows_before:,}')
print(f'rows after negative duration filter: {rows_after:,}')
print(f'rows removed: {rows_before - rows_after:,}')

rows before negative duration filter: 7,617
rows after negative duration filter: 7,519
rows removed: 98


## Step 6: Removal Summary

สรุปจำนวนแถวที่ถูกตัดออกแยกตามเหตุผลที่ตรวจในขั้นนี้

In [119]:
removal_summary = pd.DataFrame(
    {
        'reason': [
            'OperatorCarConfirm > CarConfirm',
            'CarConfirm > FirstPostPallet',
            'FirstPostPallet > LastPostPallet',
            'LastPostPallet > PostingTime',
            'wait_call_min < 0',
            'prepare_loading_min < 0',
            'loading_time_min < 0',
            'close_job_min < 0',
            'any negative duration row removed',
        ],
        'rows': [
            int(time_order_summary.loc[time_order_summary['rule'] == 'OperatorCarConfirm <= CarConfirm', 'violated_rows'].iloc[0]),
            int(time_order_summary.loc[time_order_summary['rule'] == 'CarConfirm <= FirstPostPallet', 'violated_rows'].iloc[0]),
            int(time_order_summary.loc[time_order_summary['rule'] == 'FirstPostPallet <= LastPostPallet', 'violated_rows'].iloc[0]),
            int(time_order_summary.loc[time_order_summary['rule'] == 'LastPostPallet <= PostingTime', 'violated_rows'].iloc[0]),
            int((duration_check['wait_call_min'] < 0).sum()),
            int((duration_check['prepare_loading_min'] < 0).sum()),
            int((duration_check['loading_time_min'] < 0).sum()),
            int((duration_check['close_job_min'] < 0).sum()),
            int(negative_duration_mask.sum()),
        ],
    }
)

removal_summary

,reason,rows
0,OperatorCarConfirm > CarConfirm,98
1,CarConfirm > FirstPostPallet,0
2,FirstPostPallet > LastPostPallet,0
3,LastPostPallet > PostingTime,0
4,wait_call_min < 0,98
5,prepare_loading_min < 0,0
6,loading_time_min < 0,0
7,close_job_min < 0,0
8,any negative duration row removed,98


## Preview Cleaned Data

ดูข้อมูลหลัง clean ขั้นล่าสุด

In [120]:
print(f'final shape for current step: {df_clean.shape}')
df_clean.head()

final shape for current step: (7519, 37)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,PRESTIGEFittingSapAmount,NEUSTILEFittingSapAmount,DURAFittingSapAmount,ACCESSORIESSapAmount,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd
10,SB1,Walk in,2026-04-10 08:52:23,1,4 ล้อ,84-5388,SB1PL260410015,หจก.สำรวยเซรามิค,2026-04-10 08:52:32,N,...,0,0,0,0,2026-04-10 09:23:51,2026-04-10 09:24:56,2026-04-10 09:00:14,2026-04-10 09:04:32,NaN,NaN
11,SB1,Walk in,2026-04-10 08:48:55,4,6 ล้อ,70-4573,SB1PL260410014,บริษัท วันโฮม วัสดุก่อสร้าง จำกัด,2026-04-10 08:49:00,N,...,0,0,0,0,2026-04-10 09:08:03,2026-04-10 09:13:52,NaN,NaN,NaN,NaN
12,SB1,Walk in,2026-04-10 08:47:25,1,4 ล้อ,3ฒอ9423,SB1PL260410013,บ.รวมซีเมนต์99 จก.,2026-04-10 08:47:34,N,...,50,0,0,1,2026-04-10 09:15:01,2026-04-10 09:22:14,2026-04-10 08:55:17,2026-04-10 08:58:33,2026-04-10 08:50:45,2026-04-10 08:52:05
13,SB1,Walk in,2026-04-10 08:42:51,2,4 ล้อ,บล4874,SB1PL260410012,บ.เงินทองมาวัสดุ จก.,2026-04-10 08:42:56,N,...,30,0,0,0,2026-04-10 08:55:58,2026-04-10 08:56:39,2026-04-10 08:52:42,2026-04-10 08:54:33,NaN,NaN
14,SB1,Walk in,2026-04-10 08:29:57,6,6 ล้อ,71-0658,SB1PL260410011,บริษัท วันโฮม วัสดุก่อสร้าง จำกัด,2026-04-10 08:30:02,N,...,0,0,0,0,2026-04-10 08:45:24,2026-04-10 08:46:40,NaN,NaN,NaN,NaN


## Save Current Clean File

บันทึกผล clean ปัจจุบันไว้ก่อน เพื่อใช้ต่อในขั้นถัดไป

In [121]:
output_path = INTERIM_DATA_DIR / 'vw_timestamp_dashboard_clean.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8-sig')
output_path

WindowsPath('D:/ML/data/interim/vw_timestamp_dashboard_clean.csv')